<a href="https://colab.research.google.com/github/Bachbean/Predicting-the-Unpredictable/blob/main/scripts/L63_embedding_dimension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

file_name = "https://raw.githubusercontent.com/Bachbean/Predicting-the-Unpredictable/refs/heads/main/scripts/L63_x.txt"

data = np.loadtxt(file_name, dtype=np.float64)


# ============================================================
# SETTINGS
# ============================================================

tau = 16
max_dimension = 10

# Standard false-neighbor threshold
R_threshold = 10.0


# ============================================================
# CREATE DELAY EMBEDDING
# ============================================================

def delay_embed(data, dimension, tau):

    N = len(data) - (dimension - 1) * tau

    embedded = np.empty((N, dimension))

    for j in range(dimension):
        embedded[:, j] = data[j * tau : j * tau + N]

    return embedded


# ============================================================
# CALCULATE FALSE NEAREST NEIGHBORS
# ============================================================

fnn_percentages = []

for m in range(1, max_dimension + 1):

    # We need enough points for both m and m+1 dimensions
    N = len(data) - m * tau

    if N <= 0:
        break

    Xm = delay_embed(data, m, tau)[:N]
    Xm1 = delay_embed(data, m + 1, tau)[:N]

    # Find nearest neighbors in m dimensions
    neighbors = NearestNeighbors(
        n_neighbors=2
    ).fit(Xm)

    distances, indices = neighbors.kneighbors(Xm)

    # First neighbor is the point itself
    nearest_distance = distances[:, 1]
    nearest_index = indices[:, 1]

    # Difference introduced by extra dimension
    extra_difference = np.abs(
        Xm1[:, -1] -
        Xm1[nearest_index, -1]
    )

    # Avoid dividing by zero
    valid = nearest_distance > 1e-12

    ratio = np.zeros_like(nearest_distance)

    ratio[valid] = (
        extra_difference[valid] /
        nearest_distance[valid]
    )

    false_neighbors = ratio > R_threshold

    percentage = (
        np.sum(false_neighbors[valid]) /
        np.sum(valid)
    ) * 100

    fnn_percentages.append(percentage)

    print(
        f"Dimension {m}: "
        f"{percentage:.3f}% false nearest neighbors"
    )


# ============================================================
# PLOT RESULTS
# ============================================================

dimensions = np.arange(
    1,
    len(fnn_percentages) + 1
)

plt.figure(figsize=(10, 5))

plt.plot(
    dimensions,
    fnn_percentages,
    marker="o"
)

plt.xlabel("Embedding Dimension m")
plt.ylabel("False Nearest Neighbors (%)")
plt.title("False Nearest Neighbors")

plt.grid(alpha=0.3)

plt.show()